In [79]:
# Cell 1
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [80]:
class RegressionData:
    """선형회귀 훈련 데이터를 batch로 제공한다."""
    
    def __init__(self, batch_size=2):
        self.batch_size = batch_size
        
        # sample 6개, feature 2개
        features = torch.tensor([
            [1.0, 2.0],
            [2.0, 1.0],
            [3.0, 4.0],
            [4.0, 3.0],
            [5.0, 2.0],
            [2.0, 5.0],
        ])

        # true_weights = [2, -3], true_bias = 5로 만든 label
        labels = torch.tensor([
            [1.0],
            [6.0],
            [-1.0],
            [4.0],
            [9.0],
            [-6.0],
        ])

        # TensorDataset은 feature와 label을 sample 단위로 묶는다.
        self.train_dataset = TensorDataset(features, labels)
        
    def train_dataloader(self):
        # 전체 데이터셋을 batch_size개씩 반환하는 DataLoader
        return DataLoader(
            dataset=self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
        )
    
    
    def val_dataloader(self):
        return None
    

In [81]:
# Cell 3
class LinearRegression(nn.Module):
    """feature 2개를 받아 숫자 하나를 예측하는 선형회귀 모델."""

    def __init__(self, learning_rate=0.03):
        super().__init__()

        self.learning_rate = learning_rate
        self.net = nn.Linear(
            in_features=2,
            out_features=1,
        )

    def forward(self, features):
        # model(features)를 실행하면 이 method가 호출된다.
        return self.net(features)

    def loss(self, predictions, labels):
        # sample별 제곱손실을 계산한 뒤 batch 평균을 반환한다.
        errors = predictions - labels
        per_example_loss = 0.5 * errors.pow(2)
        return per_example_loss.mean()

    def configure_optimizers(self):
        # SGD가 이 모델의 weight와 bias를 갱신하도록 설정한다.
        return torch.optim.SGD(
            self.parameters(),
            lr=self.learning_rate,
        )

In [82]:
class Trainer:
    """모델과 데이터를 연결하여 실제 학습 반복문을 실행한다."""
    
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs
        
    def prepare_data(self, data):
        # DataModule에서 training & validation DataLoader를 가져온다.
        self.train_dataloader = data.train_dataloader()
        self.val_dataloader = data.val_dataloader()

        # 한 epoch에서 처리할 훈련 batch 수
        self.num_train_batches = len(self.train_dataloader)
        
        # validation DataLoader가 없으면 validation batch는 0
        if self.val_dataloader is None:
            self.num_val_batches = 0
        else:
            self.num_val_batches = len(self.val_dataloader)
     
     
    def prepare_model(self, model):
        # Trainer가 훈련할 모델을 저장한다.
        self.model = model

        # 모델에서도 자신을 담당하는 Trainer에 접근할 수 있게 한다.
        model.trainer = self
        
        
    def fit(self, model, data):
        # 데이터와 모델을 훈련 가능한 상태로 준비한다.
        self.prepare_data(data)
        self.prepare_model(model)
        
        # 모델이 지정한 optimizer를 생성한다.
        self.optimizer = model.configure_optimizers()

        # loss 변화 기록
        self.loss_history = []
        
        # 전체 데이터셋을 max_epochs번 반복한다.
        for self.epoch in range(self.max_epochs):
            epoch_loss = self.fit_epoch()
            self.loss_history.append(epoch_loss)

            # 출력이 너무 많아지지 않도록 10 epoch마다 표시한다.
            if (
                self.epoch == 0
                or (self.epoch + 1) % 10 == 0
            ):
                print(
                    f"Epoch {self.epoch + 1:3d} "
                    f"| Loss {epoch_loss:.6f}"
                )
        
    def fit_epoch(self):
        self.model.train()

        total_loss = 0.0
        number_of_samples = 0
        
        # DataLoader가 현재 epoch의 batch를 하나씩 제공한다.
        for features, labels in self.train_dataloader:
            
            # 이전 batch에서 계산된 gradient를 제거
            self.optimizer.zero_grad()
            
            # 현재 parameter로 batch의 예측값을 계산
            predictions = self.model(features)
            
            # 현재 batch의 평균 squared loss 계산
            loss = self.model.loss(predictions, labels)
            
            # loss에서 weight와 bias까지 역전파하여
            # parameter의 gradient를 계산한다
            loss.backward()
            
            # 계산된 gradient를 사용해 parameter를 실제 갱신한다.
            self.optimizer.step()
            
            batch_size = labels.shape[0]
            
            # epoch 전체의 평균 loss를 구하기 위해
            # batch 평균 loss에 batch sample 수를 다시 곱해 누적한다.
            total_loss += loss.item() * batch_size
            number_of_samples += batch_size

        return total_loss / number_of_samples
        
        

In [83]:
# Cell 5
data = RegressionData(batch_size=2)
model = LinearRegression(learning_rate=0.03)
trainer = Trainer(max_epochs=100)

# 모델, 데이터, optimizer를 연결해 실제 훈련을 시작한다.
trainer.fit(model, data)

Epoch   1 | Loss 14.242463
Epoch  10 | Loss 1.720934
Epoch  20 | Loss 1.106352
Epoch  30 | Loss 1.133457
Epoch  40 | Loss 0.817047
Epoch  50 | Loss 0.810452
Epoch  60 | Loss 0.570664
Epoch  70 | Loss 0.450954
Epoch  80 | Loss 0.389974
Epoch  90 | Loss 0.336786
Epoch 100 | Loss 0.264701


In [84]:
# Cell 6
# 학습된 parameter를 확인한다.
# 정답은 weight=[2, -3], bias=5다.
learned_weights = model.net.weight.detach()
learned_bias = model.net.bias.detach()

print("Learned weights:")
print(learned_weights)

print("\nLearned bias:")
print(learned_bias)

print("\nFinal loss:")
print(trainer.loss_history[-1])

assert learned_weights.shape == (1, 2)
assert learned_bias.shape == (1,)
assert trainer.loss_history[-1] < trainer.loss_history[0]

Learned weights:
tensor([[ 2.3769, -2.6370]])

Learned bias:
tensor([2.7547])

Final loss:
0.2647013107004265
